# Speech Understanding with SenseVoiceSmall and OpenVINO

[SenseVoiceSmall](https://huggingface.co/FunAudioLLM/SenseVoiceSmall) is a speech understanding model from FunAudioLLM. A non-autoregressive encoder and CTC head produce a transcript together with language, emotion and sound-event tags, without an autoregressive text decoder. The released Small checkpoint supports Mandarin, Cantonese, English, Japanese and Korean.

This tutorial uses [Optimum Intel](https://huggingface.co/docs/optimum/intel/openvino/export) for conversion and [OpenVINO GenAI ASRPipeline](https://openvinotoolkit.github.io/openvino.genai/docs/use-cases/speech-recognition/) for inference. See the [original repository](https://github.com/FunAudioLLM/SenseVoice) and [paper](https://arxiv.org/abs/2407.04051) for the architecture and model evaluations.

This tutorial demonstrates how to:

- export an FP16 model or compress its weights to INT8 with NNCF;
- transcribe audio and inspect language, emotion and event predictions;
- select a language and compare inverse text normalization (ITN) settings;
- transcribe two recordings in one batch;
- try an audio file or microphone recording in Gradio.

The model weights use the [custom FunASR model license](https://github.com/modelscope/FunASR/blob/main/MODEL_LICENSE), not Apache-2.0. Review its terms before use. Export requires `trust_remote_code=True`, which permits loading model-provided Python code; the source model revision is fixed below.

This example handles recordings up to 30 seconds; long-audio segmentation, translation, speaker diarization and word timestamps are outside its scope.



### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/sensevoice-small/sensevoice-small.ipynb" />

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Installation](#Installation)
- [Select model and export options](#Select-model-and-export-options)
- [Select inference device](#Select-inference-device)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Prepare short audio examples](#Prepare-short-audio-examples)
- [Transcribe and inspect language, emotion and event](#Transcribe-and-inspect-language,-emotion-and-event)
- [Select a language and compare ITN](#Select-a-language-and-compare-ITN)
- [Transcribe a batch](#Transcribe-a-batch)
- [Interactive demo](#Interactive-demo)
- [Clean up](#Clean-up)



## Prerequisites
[back to top ⬆️](#Table-of-contents:)

### Installation
[back to top ⬆️](#Table-of-contents:)

Install the required dependencies:

- Optimum Intel is installed from `main` for [SenseVoice export support](https://github.com/huggingface/optimum-intel/pull/2038) and [GPU activation scaling](https://github.com/huggingface/optimum-intel/pull/2118).
- OpenVINO, Tokenizers and GenAI use pre-release packages from the nightly index for [ASR](https://github.com/openvinotoolkit/openvino.genai/pull/4592), [metadata](https://github.com/openvinotoolkit/openvino.genai/pull/4625) and [batching](https://github.com/openvinotoolkit/openvino.genai/pull/4701) support. This notebook targets OpenVINO 2026.5.
- Transformers stays in the 4.57.x range exercised by the upstream ASR tests. Matching PyTorch and TorchAudio 2.8.x packages provide the export and audio frontend.

The original checkpoint download is approximately 0.94 GB. Export stores another copy in OpenVINO format. This notebook installs CPU PyTorch wheels; OpenVINO handles GPU inference independently of PyTorch.

Restart the notebook kernel after installing or changing these packages.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "cmd_helper.py", "pip_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "--pre",
    "git+https://github.com/huggingface/optimum-intel.git@main",
    "openvino>=2026.5.1.dev20261113",
    "openvino-tokenizers>=2026.5.1.0.dev20261113",
    "openvino-genai>=2026.5.1.0.dev20261113",
    "nncf>=3.4.0",
    "transformers>=4.57.6,<4.58",
    "torch>=2.8,<2.9",
    "torchaudio>=2.8,<2.9",
    "funasr>=1.4.16,<1.5",
    "protobuf>=7.36.1",
    "gradio>=5,<6",
    "ipywidgets",
    "datasets>=2.19,<4",
    "librosa>=0.10",
    "soundfile>=0.12.1",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "--extra-index-url",
    "https://storage.openvinotoolkit.org/simple/wheels/nightly",
)

import ipywidgets as widgets
from IPython.display import display

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("sensevoice-small.ipynb")

## Select model and export options
[back to top ⬆️](#Table-of-contents:)

This notebook uses the official Small checkpoint. FP16 stores floating-point weights. INT8 uses NNCF weight compression without a calibration dataset; it is not full INT8 activation quantization. Change the selection and rerun from this section to compare exports.

In [ ]:
MODEL_ID = "FunAudioLLM/SenseVoiceSmall"
MODEL_REVISION = "3847d57b6bdf2dd8875cb1508d2af43d80a16bf7"

weight_format = widgets.Dropdown(options=["fp16", "int8"], value="fp16", description="Weights:")
weight_format

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select CPU or an available Intel GPU. NPU is excluded until a SenseVoice-specific export and inference path has been validated; dynamic CTC encoder shapes must not be treated as an LLM/NPU configuration.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU", "AUTO"])
device

In [ ]:
precision = weight_format.value
model_dir = Path("models") / MODEL_ID.split("/")[-1] / MODEL_REVISION[:8] / "asf4" / precision
print(f"Model: {MODEL_ID}, weights: {precision}, OpenVINO IR directory: {model_dir}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Download the reviewed source revision and export its local directory with the Optimum Intel Python API. The `OVModelForSpeechSeq2Seq` entry point also handles the single stateless SenseVoiceSmall encoder/CTC graph. Its CMVN statistics (`am.mvn`) and OpenVINO detokenizer are also required by GenAI; keep them beside the model XML/BIN files.

The exporter stores `ACTIVATIONS_SCALE_FACTOR=4.0` in the model's runtime options to prevent FP16 activation overflow on GPU. The `asf4` subdirectory keeps these exports separate from older models without this setting; upgrading Optimum Intel alone does not update an existing IR.

In [ ]:
from huggingface_hub import snapshot_download
from optimum.intel import OVModelForSpeechSeq2Seq

required_assets = ["openvino_model.xml", "openvino_model.bin", "openvino_detokenizer.xml", "openvino_detokenizer.bin", "config.json", "am.mvn"]
if not all((model_dir / name).is_file() for name in required_assets):
    source_dir = snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, allow_patterns=["*.pt", "*.yaml", "*.json", "*.mvn", "*.model", "*.py"])
    exported_model = OVModelForSpeechSeq2Seq.from_pretrained(
        str(source_dir), export=True, compile=False, load_in_8bit=precision == "int8", trust_remote_code=True
    )
    if precision == "fp16":
        exported_model.half()
    exported_model.save_pretrained(model_dir)
    del exported_model

missing_assets = [name for name in required_assets if not (model_dir / name).is_file()]
if missing_assets:
    raise FileNotFoundError(f"Incomplete SenseVoice export in {model_dir}: {missing_assets}")
print(f"OpenVINO model ready: {model_dir}")

## Prepare short audio examples
[back to top ⬆️](#Table-of-contents:)

Use two recordings from the small [LibriSpeech test fixture](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy), derived from [LibriSpeech](https://huggingface.co/datasets/openslr/librispeech_asr) (CC BY 4.0). The fixture is approximately 9 MB; the full speech corpus is not downloaded. The second utterance says "ten seconds" for the ITN comparison.

GenAI expects mono float32 audio at 16 kHz. The shared demo helper converts stereo audio to mono and changes the sample rate when needed. Listen to the samples and compare the predictions with their reference transcripts.

In [ ]:
from datasets import load_dataset
from IPython.display import Audio
from notebook_utils import download_file

if not Path("gradio_helper.py").is_file():
    download_file(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/sensevoice-small/gradio_helper.py",
        filename="gradio_helper.py",
    )
from gradio_helper import prepare_audio

AUDIO_SECONDS = 30
samples = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation", revision="5be91486e11a2d616f4ec5db8d3fd248585ac07a").select(
    [0, 56]
)

waveforms = []
for sample in samples:
    audio = sample["audio"]
    waveform = prepare_audio(waveform=audio["array"][: AUDIO_SECONDS * audio["sampling_rate"]], sampling_rate=audio["sampling_rate"])
    waveforms.append(waveform)
    print(f"Reference transcript: {sample['text']}")
    display(Audio(data=waveform, rate=16000))

## Transcribe and inspect language, emotion and event
[back to top ⬆️](#Table-of-contents:)

Create an `ASRPipeline` and use `language="auto"` for language identification. `texts[i]`, `languages[i]` and, when present, `metadata[i]` describe the same input. Emotion/event keys are optional predictions, not an emotion accuracy evaluation or a time-aligned event annotation. Inference uses the selected device's default runtime precision.

Read speech may produce neutral emotion and speech-event tags. In the interactive demo, try a short expressive recording to explore other predictions. This is an offline, single-pass CTC model; a text callback would not turn it into live acoustic streaming.

In [ ]:
import openvino_genai as ov_genai

pipe = ov_genai.ASRPipeline(str(model_dir), device.value)
single_results = []
for waveform in waveforms:
    result = pipe.generate(waveform, language="auto", use_itn=True)
    single_results.append(result)
    print(f"TRANSCRIPTION: {result.texts[0]}")
    print(f"LANGUAGE: {result.languages[0]}")
    print(f"METADATA: {result.metadata}")
    print()

## Select a language and compare ITN
[back to top ⬆️](#Table-of-contents:)

Use a language code (`en`, `zh`, `yue`, `ja` or `ko`) to select a language explicitly. ITN controls written formatting such as spoken numbers: compare `use_itn=False` with `True` on the same English recording. Exact formatting depends on the utterance and model predictions.

In [ ]:
for use_itn in (False, True):
    result = pipe.generate(waveforms[1], language="en", use_itn=use_itn)
    print(f"ITN={use_itn}: {result.texts[0]} (language: {result.languages[0]})")

## Transcribe a batch
[back to top ⬆️](#Table-of-contents:)

Pass a list of waveforms to transcribe multiple recordings in one inference. Results preserve input order, and the same language/ITN settings apply to the entire batch. Compare the output with the individual calls above: numerical differences can change individual tokens, so exact transcript equality is not guaranteed. Padding to the longest input means batching is not an unconditional speedup; group similarly sized recordings for throughput experiments.

In [ ]:
batch_result = pipe.generate(waveforms, language="auto", use_itn=True)
if len(batch_result.texts) != len(waveforms) or len(batch_result.languages) != len(waveforms):
    raise RuntimeError("The batch result does not contain one transcript and language per input.")
if batch_result.metadata is not None and len(batch_result.metadata) != len(waveforms):
    raise RuntimeError("The batch metadata is not aligned with the inputs.")

for index, text in enumerate(batch_result.texts):
    print(f"Batch item {index}: {text}")
    print(f"Same transcript as the single call: {text == single_results[index].texts[0]}")
    print(f"Language: {batch_result.languages[index]}")
    if batch_result.metadata is not None:
        print(f"Metadata: {batch_result.metadata[index]}")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Upload a short recording or record from your microphone, then submit it for transcription. A microphone is optional. Inputs longer than 30 seconds are rejected rather than silently truncated. The selected model, precision and device are reused. Interrupt the cell to stop the demo. If local launch fails, it retries with a public Gradio share link.

In [ ]:
run_demo = widgets.Checkbox(value=True, description="Launch interactive demo")
run_demo

In [ ]:
from gradio_helper import make_demo

if run_demo.value:
    demo = make_demo(pipe=pipe)

    # if you are launching remotely, specify server_name and server_port
    # demo.launch(server_name="your server name", server_port=7860)
    # Read more in the docs: https://gradio.app/docs/
    try:
        demo.launch(debug=True)
    except Exception as error:
        print(f"Local Gradio launch failed: {error}. Retrying with a public share link.")
        demo.launch(debug=True, share=True)

## Clean up
[back to top ⬆️](#Table-of-contents:)

Uncomment the relevant lines to close the demo, release the pipeline or remove this export. The source checkpoint and speech fixture remain in their Hugging Face caches and can be managed separately.

In [ ]:
# demo.close()
# del pipe

# import shutil
# shutil.rmtree(model_dir)